# Volume 1. Operations Side By Side

Question: how do projection, reciprocal projection, association, merge, and pattern completion differ when inspected with the same scene?

The goal is a compact comparison. Each operation is run in its own fresh brain so that one demo does not contaminate the next.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML

from neural_assemblies.assembly_calculus import project
from neural_assemblies.assembly_calculus.tracing import (
    associate_trace,
    merge_trace,
    pattern_complete_trace,
    project_trace,
    reciprocal_project_trace,
)
from neural_assemblies.core.brain import Brain
from neural_assemblies.viz import animate_assembly_trace, plot_trace_metrics, plot_winner_turnover


In [ ]:
N = 4_000
K = 60
BETA = 0.08
SEED = 23


def make_scene():
    brain = Brain(p=0.05, save_winners=True, seed=SEED, engine="numpy_sparse")
    brain.add_stimulus("red", K)
    brain.add_stimulus("triangle", K)
    for area in ["COLOR", "SHAPE", "COPY", "TARGET"]:
        brain.add_area(area, N, K, BETA)
    project(brain, "red", "COLOR", rounds=7)
    project(brain, "triangle", "SHAPE", rounds=7)
    return brain


Run each operation and collect the same kind of diagnostic row: operation, target, rounds, final winner count, and stabilization behavior.


In [ ]:
projection_brain = make_scene()
projection = project_trace(projection_brain, "red", "TARGET", rounds=6)

copy_brain = make_scene()
copy_trace = reciprocal_project_trace(copy_brain, "COLOR", "COPY", rounds=6)

association_brain = make_scene()
association = associate_trace(association_brain, "COLOR", "SHAPE", "TARGET", rounds=3)

merge_brain = make_scene()
merged = merge_trace(merge_brain, "COLOR", "SHAPE", "TARGET", rounds=5)

completion_brain = make_scene()
completion = pattern_complete_trace(completion_brain, "COLOR", fraction=0.45, rounds=5, seed=3)

summary = pd.DataFrame(
    [projection.summary(), copy_trace.summary(), association.summary(), merged.summary()]
    + [completion.to_record() | {"operation": "pattern_complete", "target": "COLOR"}]
)
summary


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6.8))
plot_trace_metrics(projection, axes=axes[0], title="Project", color="#4d9de0")
plot_trace_metrics(merged, axes=axes[1], title="Merge", color="#59a14f")
plt.show()
plt.close(fig)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
plot_winner_turnover(copy_trace, ax=axes[0], title="Reciprocal projection")
plot_winner_turnover(association, ax=axes[1], title="Association phases")
plot_winner_turnover(completion.trace, ax=axes[2], title="Pattern completion")
plt.show()
plt.close(fig)


Animate one operation that is easy to misread as a static before/after: merge. The target is being driven by two fixed source assemblies over multiple rounds.


In [ ]:
fig, animation = animate_assembly_trace(
    merged,
    N,
    title="Merge target dynamics",
    color="#59a14f",
    interval=750,
)
html = HTML(animation.to_jshtml())
plt.close(fig)
html


Side-by-side inspection makes the primitive boundaries clearer: projection forms a target from external drive, reciprocal projection copies across areas, association creates sequential shared access, merge binds simultaneous sources, and pattern completion tests recurrence as an attractor.
